# ERK-KTR Full FOV Stimulation -- Pre-Exposure Drug Test (df_acquire API)

This notebook runs a multi-phase ERK-KTR optogenetic experiment with full-FOV light stimulation on the Jungfrau microscope.

**API used:** This notebook uses the **legacy `df_acquire` DataFrame API** to define acquisition events. The DataFrame is built with `generate_df_acquire()` + `apply_stim_treatments_to_df_acquire()`, then converted to RTMEvents via `df_to_events()` for execution. For the newer RTMSequence-based API, see `Full_FOV_stim_ERKKTR_RTMSequence.ipynb`.

**Experimental design:**
- 3 FOV groups (each ~21 FOVs) receive different stimulation treatment sets
- Each group is split at timestep 100 into pre-drug and post-drug halves
- Drug is pipetted manually between phases (the notebook pauses at cell boundaries)
- Stimulation patterns vary by drug concentration, pre-stim status, and timing

**Workflow overview:**
1. Initialize microscope and define channels
2. Define stimulation treatments (frequencies, exposures, drug concentrations)
3. Configure the image processing pipeline
4. Select FOV positions in napari
5. Generate `df_acquire` DataFrames for each FOV group and split into pre/post-drug phases
6. Run the experiment in sequential phases with manual drug addition between them
7. Post-process tracks into `exp_data.parquet`

### Import required libraries

In [ ]:
import os
import time
from rtm_pymmcore.core.data_structures import (
    Channel,  # basic imaging channel (config, exposure, group)
    PowerChannel,  # imaging channel with light-source power control (adds power 0-100)
    StimTreatment,  # defines a stimulation treatment (timesteps, exposure, power, channel)
    SegmentationMethod,  # wraps a segmentator with channel index and label name
)
import rtm_pymmcore.core.utils as utils
from pprint import pprint
import pandas as pd

### Experimental Settings

**Microscope:** Jungfrau (no DMD -- full-FOV stimulation only).

**Channel types:**
- `Channel(config, exposure, group)` -- basic imaging channel
- `PowerChannel(config, exposure, group, power)` -- adds light source power control (0-100)

**What to customize below:**
- `N_FRAMES` / `TIME_BETWEEN_TIMESTEPS` -- time-lapse schedule per phase
- `base_path` / `experiment_name` -- output directory
- `channels` -- imaging channels acquired every timepoint (order matters: channel 0 is used for segmentation)
- `channel_optocheck` -- reference channel to verify optogenetic tool expression
- `condition` -- experimental condition labels assigned to FOVs
- `n_fovs_per_well` -- wellplate layout (set `None` for free-FOV experiments)

In [ ]:
from rtm_pymmcore.microscope.pertzlab.jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup(
    "TTL_ERK"
)  # select the channel group configured in Micro-Manager

In [ ]:
N_FRAMES = (
    180  # total timesteps per FOV group (split at timestep 100 into pre/post-drug)
)

TIME_BETWEEN_TIMESTEPS = 60  # seconds between consecutive timesteps
TIME_PER_FOV = 2  # seconds per FOV (camera exposure + stage movement + overhead)

# df_acquire options for stimulation treatment assignment
ADD_STIM_EXPOSURE_GROUP = (
    False  # if True, saves per-FOV last-stim-exposure info in the DataFrame
)
REGULAR_SPACING_BETWEEN_STIMULATIONS = (
    False  # True = evenly spaced stim; False = use explicit stim_timestep lists
)

## Storage -- all output (zarr, tracks, TIFFs) goes under this directory
base_path = "E:\\Alex"
experiment_name = "2026-04-02_SUSTAINED_DIFF.DENS_RMC6236"
path = os.path.join(base_path, experiment_name)

## Imaging channels -- acquired at every timepoint; order matters (channel 0 is used for segmentation)
channels = []
channels.append(
    PowerChannel(
        config="miRFP", exposure=125, group="TTL_ERK", power=95
    )  # nuclear marker
)
channels.append(
    PowerChannel(
        config="mScarlet3", exposure=125, group="TTL_ERK", power=95
    )  # ERK-KTR reporter
)

## Optocheck channel -- reference channel to verify optogenetic tool expression (longer exposure)
channel_optocheck = PowerChannel(
    config="mCitrine", exposure=600, group="TTL_ERK", power=99
)
optocheck_timepoints = (N_FRAMES - 1,)  # acquire optocheck only on the last frame

## Experimental conditions -- labels assigned to FOV groups for treatment mapping
condition = [
    "EGFR",
]

## Wellplate layout -- set how many FOVs per well; None for free-FOV experiments
n_fovs_per_well = 3

### Stimulation Treatments

Each `StimTreatment` defines when and how a group of FOVs is stimulated:
- `treatment_name` -- human-readable label (parsed later for analysis)
- `stim_timestep` -- tuple of frame indices that receive stimulation
- `stim_exposure_list` -- exposure duration(s) in ms (single value or per-frame tuple for ramps)
- `stim_power` -- light source intensity (0-100)
- `stim_channel_*` -- Micro-Manager channel/device configuration for the stimulation light
- `auto_repeat_stim_exposure` -- if True, cycles through exposure list when it is shorter than stim_timestep

**Timestep library:** Pre-defined frame ranges for common timing patterns. `with_prestim` includes frames 10-99 (pre-drug stimulation) plus 101-159 (post-drug). `without_prestim` only includes 101-159.

**Helper functions:**
- `build_stim_treatments()` -- constructs `StimTreatment` objects from compact specs
- `parse_treatment_name()` -- extracts drug concentration, exposure, and pre-stim status from the treatment name string
- `split_df_acquire()` -- splits a df_acquire at a given timestep for multi-phase experiments (resets time offset for the second half)

In [ ]:
# Define frame ranges for stimulation timing patterns
# with_prestim: stimulate during pre-drug (frames 10-99) AND post-drug (frames 101-159)
with_prestim_timesteps = tuple(range(10, 100, 1)) + tuple(range(101, 160, 1))
# without_prestim: stimulate only post-drug (frames 101-159)
without_prestim_timesteps = tuple(range(101, 160, 1))
import numpy as np

# Reusable timestep library -- maps (prestim_sign, interval_label) to frame tuples
timestep_lookup = {
    ("+", "1min"): with_prestim_timesteps,  # with pre-drug stimulation, 1 min interval
    (
        "-",
        "1min",
    ): without_prestim_timesteps,  # without pre-drug stimulation, 1 min interval
}


def build_stim_treatments(specs, timestep_lookup, *, stim_power=10):
    """Build a list of StimTreatment objects from compact row-like specs.

    Each spec is a tuple: (drug_nM, prestim_bool, timing_key, exposure_ms)
      - drug_nM: drug concentration label (used in treatment_name)
      - prestim_bool: True = stimulate pre-drug (+), False = post-drug only (-)
      - timing_key: key into timestep_lookup (e.g. "1min")
      - exposure_ms: stimulation exposure in ms, or "ru" for ramp-up protocol
    """
    treatments = []
    for drug_uM, prestim, timing_key, exposure_ms in specs:
        sign = "+" if prestim else "-"
        key = (sign, timing_key)
        if key not in timestep_lookup:
            raise ValueError(f"Unknown timing config: {key}")

        if exposure_ms != "ru" and not isinstance(exposure_ms, int):
            raise ValueError(f"Exposure must be 'ru' or int, got {exposure_ms}")

        # "ru" = ramp-up: linearly increasing exposure pre-drug, then constant post-drug
        if exposure_ms == "ru":
            if prestim:
                exposure_ms = tuple(range(0, 900, 10)) + (250,) * 60
            else:
                exposure_ms = (250,) * 60
            treatment_name = f"{drug_uM}nM -1ms {sign} stim{timing_key}"
        else:
            treatment_name = f"{drug_uM}nM {exposure_ms}ms {sign} stim{timing_key}"

        print(treatment_name)
        treatments.append(
            StimTreatment(
                treatment_name=treatment_name,
                stim_timestep=timestep_lookup[key],
                stim_exposure_list=exposure_ms,
                stim_power=stim_power,
                stim_channel_name="CyanStim",
                stim_channel_group="TTL_ERK",
                stim_channel_device_name="Spectra",
                stim_channel_power_property_name="Cyan_Level",
                auto_repeat_stim_exposure=True,  # cycle exposure list if shorter than stim_timestep
            )
        )
    return treatments


def parse_treatment_name(df_acquire):
    """Parse treatment_name column into separate columns for analysis.

    Expected format: "{drug_nM}nM {exposure}ms {+/-} stim{interval}"
    Produces columns: drug_concentration, pattern_exposure, prestimulation, stim_every_x_min
    """
    parts = df_acquire["treatment_name"].str.split(" ", expand=True)
    df_acquire["drug_concentration"] = parts[0].str.replace("nM", "").astype(int)
    if "ru" in parts[1].iloc[0]:
        df_acquire["pattern_exposure"] = np.nan
    else:
        df_acquire["pattern_exposure"] = parts[1].str.replace("ms", "").astype(float)
    df_acquire["prestimulation"] = parts[2] == "+"
    df_acquire["stim_every_x_min"] = parts[3].str.replace("stim", "")
    df_acquire["stim_every_x_min"] = (
        df_acquire["stim_every_x_min"].str.replace("min", "").astype(int)
    )
    return df_acquire


def split_df_acquire(df, split_timestep):
    """Split df_acquire at split_timestep into two halves.

    The second half gets its time column reset so it starts at 0.
    Used to insert a manual drug-addition step between pre- and post-drug phases.
    """
    t_offset = df.loc[df["timestep"] == split_timestep, "time"].iloc[0]
    df_1 = df[df["timestep"] < split_timestep]
    df_2 = df[df["timestep"] >= split_timestep].copy()
    df_2["time"] = (
        df_2["time"] - t_offset
    )  # reset time so post-drug phase starts at t=0
    return df_1, df_2

In [ ]:
# Define stimulation treatment specs for each FOV group.
# Each spec: (drug_nM, prestim(+/-), timing_key, exposure_ms)
# FOVs are divided into 3 groups of ~21, each assigned a different treatment set.

# Group 1: FOVs 0-20
stim_phase_1 = [
    (300, False, "1min", 250),  # 300 nM, no pre-drug stim, 250 ms pulse
    (300, True, "1min", 250),  # 300 nM, with pre-drug stim, 250 ms pulse
    (500, False, "1min", 250),
    (500, True, "1min", 250),
    (0, False, "1min", 250),  # control (0 nM)
    (0, True, "1min", 250),
    (300, False, "1min", 250),
]
stim_phase_1 = build_stim_treatments(stim_phase_1, timestep_lookup)

# Group 2: FOVs 21-41
stim_phase_2 = [
    (300, True, "1min", 250),
    (500, False, "1min", 250),
    (500, True, "1min", 250),
    (0, False, "1min", 250),
    (0, True, "1min", 250),
    (300, False, "1min", 250),
    (500, False, "1min", 250),
]
stim_phase_2 = build_stim_treatments(stim_phase_2, timestep_lookup)

# Group 3: FOVs 42+
stim_phase_3 = [
    (0, False, "1min", 250),
    (300, True, "1min", 250),
    (500, True, "1min", 250),
    (300, False, "1min", 250),
    (500, False, "1min", 250),
    (300, True, "1min", 250),
    (500, True, "1min", 250),
]
stim_phase_3 = build_stim_treatments(stim_phase_3, timestep_lookup)

In [ ]:
## Define the image processing pipeline components.
## These run asynchronously during acquisition. All components are optional except storage_path.
## For details on each component, see Full_FOV_stim_ERKKTR_RTMSequence.ipynb.

from rtm_pymmcore.stimulation.base import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck import OptoCheckFE
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    SegmentationMethod(
        name="labels",  # label layer name in the zarr store
        segmentation_class=CellposeV4(),  # Cellpose v4 with default model
        use_channel=0,  # segment on first imaging channel (miRFP)
        save_tracked=True,  # persist tracked label masks alongside raw segmentation
    )
]

stimulator = StimWholeFOV()  # illuminate entire FOV (no DMD patterning)
feature_extractor = FE_ErkKtr("labels")  # cytoplasmic/nuclear ratio using "labels" mask
tracker = TrackerTrackpy(search_range=50)  # max 50 px displacement between frames
optocheck = OptoCheckFE(used_mask="labels")  # measure optogenetic reporter per cell

from rtm_pymmcore.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_ref=optocheck,  # runs only on optocheck timepoints
)

from rtm_pymmcore.core.controller import Controller
from rtm_pymmcore.core.writers import OmeZarrWriter
from rtm_pymmcore.core.conversion import (
    df_to_events,
)  # converts df_acquire DataFrame -> RTMEvent list

writer = OmeZarrWriter(storage_path=path)  # saves images + stim readout into OME-Zarr

### GUI -- Napari + Micro-Manager

Opens a napari viewer with the Micro-Manager widget. Use this to:
- Live-view the camera feed
- Select FOV positions using the MDA widget (positions are read by `generate_fov_positions` later)

#### Load GUI

In [ ]:
### Launch napari with the Micro-Manager widget ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc  # point the widget at our CMMCore instance
viewer.window.add_dock_widget(mm_wdg)  # dock Micro-Manager controls in napari

#### Sort & Reorder FOV positions

Sort FOV positions by well columns instead of rows. 

In [ ]:
# Read positions from the napari MDA widget and sort by well name
fov_pos = viewer.window.dock_widgets["MDA"].value().stage_positions


def sort_key(fov):
    """Sort FOVs by well number, then letter, then index within well."""
    name = fov.name
    well, idx = name.split("_", 1)
    letter = well[0]
    number = int(well[1:])
    return (number, letter, idx)


fovs_sorted = sorted(fov_pos, key=sort_key)

# Update the MDA widget with the sorted positions
current_widget_values = viewer.window.dock_widgets["MDA"].value()
updated_widget = current_widget_values.model_copy(
    update={"stage_positions": fovs_sorted}
)
viewer.window.dock_widgets["MDA"].setValue(updated_widget)

#### Plot FOV positions

Visualize selected FOV positions to verify spatial layout before starting the experiment.

In [ ]:
# Read FOV positions from napari MDA widget into rtm-pymmcore FovPosition objects
fovs = utils.generate_fov_positions(mic, viewer=viewer)

In [ ]:
import matplotlib.pyplot as plt

xs = [f.x for f in fovs]
ys = [f.y for f in fovs]

plt.figure(figsize=(8, 6))
plt.scatter(xs, ys)
for fovi in fovs:
    plt.annotate(
        fovi.index,
        (fovi.x, fovi.y),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=7,
    )
plt.xlabel("X")
plt.ylabel("Y")
plt.title("FOV positions")
plt.axis("equal")
plt.tight_layout()
plt.show()
print(f"{len(fovs)} FOVs selected")

### Generate df_acquire DataFrames

The `df_acquire` DataFrame is the legacy way to define acquisition events. Each row represents one event (one channel at one timepoint at one FOV). The workflow is:

1. `generate_df_acquire()` -- creates the base DataFrame with timing, channels, and FOV positions
2. `apply_stim_treatments_to_df_acquire()` -- assigns stimulation treatments to FOV groups
3. `parse_treatment_name()` -- extracts structured metadata from the treatment name string
4. `split_df_acquire()` -- splits at a timestep boundary for multi-phase experiments
5. `df_to_events()` -- converts the DataFrame to a list of `RTMEvent` objects for the Controller

Each FOV group (0-20, 21-41, 42+) gets its own df_acquire with a different treatment set.

#### FOV Group 1 (FOVs 0-20)

In [ ]:
# Re-read FOV positions (in case they were updated in the GUI)
fovs = utils.generate_fov_positions(mic, viewer=viewer)

In [ ]:
# Generate df_acquire for FOV group 1 (first 21 FOVs)
print(f"FOV group 1: {len(fovs[0:21])} FOVs")

df_acquire_1 = utils.generate_df_acquire(
    fovs[0:21],  # FOV subset for this group
    n_frames=N_FRAMES,  # total timesteps (180)
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,  # interval between frames (60s)
    time_per_fov=TIME_PER_FOV,  # time budget per FOV (2s) -- used for batching
    channels=channels,  # imaging channels (miRFP, mScarlet3)
    condition=condition,  # condition labels for treatment assignment
    optocheck_timepoints=optocheck_timepoints,  # frames where optocheck channel is acquired
    channel_optocheck=channel_optocheck,  # optocheck channel definition
)

# Assign stimulation treatments to FOVs based on condition and n_fovs_per_well
df_acquire_1 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_1,
    stim_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)

# Parse treatment_name into structured columns (drug_concentration, prestimulation, etc.)
df_acquire_1 = parse_treatment_name(df_acquire_1)

# Split at timestep 100: pre-drug (1_1) and post-drug (1_2)
df_acquire_1_1, df_acquire_1_2 = split_df_acquire(df_acquire_1, 100)

#### FOV Group 2 (FOVs 21-41)

In [ ]:
# Generate df_acquire for FOV group 2 (FOVs 21-41)
print(f"FOV group 2: {len(fovs[21:21+21])} FOVs")

df_acquire_2 = utils.generate_df_acquire(
    fovs[21 : 21 + 21],
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    condition=condition,
    optocheck_timepoints=optocheck_timepoints,
    channel_optocheck=channel_optocheck,
)

df_acquire_2 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_2,
    stim_phase_2,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)

df_acquire_2 = parse_treatment_name(df_acquire_2)
df_acquire_2_1, df_acquire_2_2 = split_df_acquire(df_acquire_2, 100)

#### FOV Group 3 (FOVs 42+)

In [ ]:
# Generate df_acquire for FOV group 3 (remaining FOVs from index 42 onward)
df_acquire_3 = utils.generate_df_acquire(
    fovs[21 + 21 :],
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    condition=condition,
    optocheck_timepoints=optocheck_timepoints,
    channel_optocheck=channel_optocheck,
)

df_acquire_3 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_3,
    stim_phase_3,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)

df_acquire_3 = parse_treatment_name(df_acquire_3)
df_acquire_3_1, df_acquire_3_2 = split_df_acquire(df_acquire_3, 100)

### Run experiment

The experiment is run in 6 sequential sub-phases (3 FOV groups x 2 halves each). Drug is pipetted manually between pre-drug and post-drug halves.

**Execution order:**
1. `df_acquire_1_1` -- Group 1 pre-drug (wait for scheduled start time)
2. *--- Manually pipette drug for group 1 ---*
3. `df_acquire_1_2` + `df_acquire_2_1` -- Group 1 post-drug + Group 2 pre-drug
4. *--- Manually pipette drug for group 2 ---*
5. `df_acquire_2_2` + `df_acquire_3_1` -- Group 2 post-drug + Group 3 pre-drug
6. *--- Manually pipette drug for group 3 ---*
7. `df_acquire_3_2` -- Group 3 post-drug + finish

**Important:** The first phase uses `ctrl.run_experiment()`. All subsequent phases use `ctrl.continue_experiment()` to preserve pipeline state (tracking, timestep counters, filenames).

`df_to_events()` converts each `df_acquire` DataFrame into a list of `RTMEvent` objects that the Controller can execute.

In [ ]:
from datetime import datetime
import time

## Optional: schedule experiment start to align with drug pipetting
start_at = "2026-04-02 08:45:00"  # target time for 1st drug addition

# Wait so that df_acquire_1_1 finishes right at start_at (drug pipetting time)
wait_seconds = (
    datetime.strptime(start_at, "%Y-%m-%d %H:%M:%S") - datetime.now()
).total_seconds() - df_acquire_1_1.time.max()
if wait_seconds > 0:
    print(
        f"Waiting {wait_seconds/60:.1f} minutes, 1st pipetting scheduled at {start_at}"
    )
    for _ in range(0, int(wait_seconds)):
        time.sleep(1)

print("Starting experiment")

# Disconnect napari live view so it does not interfere with the MDA engine
try:
    mm_wdg._core_link.cleanup()
except:
    pass

## Phase 1: Group 1 pre-drug -- uses run_experiment (creates a new Analyzer)
ctrl = Controller(mic, pipeline, writer=writer)
events = df_to_events(df_acquire_1_1)
ctrl.run_experiment(events, stim_mode="current")

# Reconnect napari live view (pause point: pipette drug for group 1)
from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

In [ ]:
## Phase 2: Group 1 post-drug + Group 2 pre-drug
## Uses continue_experiment to preserve tracking state from phase 1
try:
    mm_wdg._core_link.cleanup()
except:
    pass

events = df_to_events(df_acquire_1_2)
ctrl.continue_experiment(events, stim_mode="current")  # preserves pipeline state

events = df_to_events(df_acquire_2_1)
ctrl.continue_experiment(events, stim_mode="current")

# Reconnect napari (pause point: pipette drug for group 2)
from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

In [ ]:
## Phase 3: Group 2 post-drug + Group 3 pre-drug
try:
    mm_wdg._core_link.cleanup()
except:
    pass

events = df_to_events(df_acquire_2_2)
ctrl.continue_experiment(events, stim_mode="current")

events = df_to_events(df_acquire_3_1)
ctrl.continue_experiment(events, stim_mode="current")

# Reconnect napari (pause point: pipette drug for group 3)
from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

In [ ]:
# Optional: manually reconnect napari if needed between phases
if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

In [ ]:
## Phase 4: Group 3 post-drug (final phase)
try:
    mm_wdg._core_link.cleanup()
except:
    pass

events = df_to_events(df_acquire_3_2)
ctrl.continue_experiment(events, stim_mode="current")

## Finish: flush pipeline queue, close zarr store, merge tracks
ctrl.finish_experiment()
utils.generate_exp_data_from_tracks(path)  # merge per-FOV tracks into exp_data.parquet

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

In [ ]:
print("Experiment complete!")

In [ ]:
## Fallback: run finish + post-processing manually if the cell above was interrupted
ctrl.finish_experiment()
utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Utility: Re-connect / Disconnect GUI

Use these cells to manually manage the connection between napari and pymmcore. Normally you don't need to run them -- the experiment cells handle this automatically.

In [ ]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()